# Video → H265 Transcoder

Reads `.mkv` / `.mp4` files from `gdrive:videototranscode`, converts to H265, saves to `gdrive:transcoded`.

**Before running:** Runtime → Change runtime type → T4 GPU (for fast NVENC encoding)

In [ ]:
# ── Cell 1: Mount Drive + install ffmpeg ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, shutil
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)

print('ffmpeg:', shutil.which('ffmpeg'))

In [ ]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────────
# Adjust these paths if your Drive folders are named differently
INPUT_DIR  = '/content/drive/MyDrive/videototranscode'
OUTPUT_DIR = '/content/drive/MyDrive/transcoded'

# CRF / CQ quality: lower = better quality, larger file.
# 24–28 is a good range; 28 ≈ visually lossless for most content.
QUALITY = 28

# Audio: 'copy' keeps the original track untouched (fastest, no quality loss).
# Change to e.g. 'aac' or 'libopus' if you want to re-encode audio too.
AUDIO_CODEC = 'copy'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Input:  {INPUT_DIR}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Detect encoder (GPU > CPU fallback) ────────────────────────────────
import subprocess

def _encoder_works(enc):
    r = subprocess.run(
        ['ffmpeg', '-hide_banner', '-loglevel', 'error',
         '-f', 'lavfi', '-i', 'nullsrc=s=64x64:d=1',
         '-c:v', enc, '-f', 'null', '-'],
        capture_output=True
    )
    return r.returncode == 0

if _encoder_works('hevc_nvenc'):
    VIDEO_CODEC = 'hevc_nvenc'
    QUALITY_FLAG = ['-cq', str(QUALITY), '-preset', 'p4', '-tier', 'high']
    print('Encoder: hevc_nvenc (GPU) ✓')
else:
    VIDEO_CODEC = 'libx265'
    QUALITY_FLAG = ['-crf', str(QUALITY), '-preset', 'medium']
    print('Encoder: libx265 (CPU) — consider switching to a GPU runtime for speed')

In [ ]:
# ── Cell 4: Helper functions ───────────────────────────────────────────────────
import json, os

def get_video_codec(path):
    """Return the video codec name (e.g. 'h264', 'hevc') via ffprobe."""
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream=codec_name',
         '-of', 'json', path],
        capture_output=True, text=True
    )
    try:
        return json.loads(r.stdout)['streams'][0]['codec_name']
    except (KeyError, IndexError, json.JSONDecodeError):
        return None

def transcode(src, dst):
    """
    Convert src → dst with H265. Returns True on success.
    Mirrors the shell script: skip if dst exists, verify output, keep src on failure.
    """
    name = os.path.basename(src)

    if os.path.exists(dst):
        print(f'[SKIP]  {name}  (output already exists)')
        return True

    codec = get_video_codec(src)
    if codec in ('hevc', 'h265'):
        print(f'[SKIP]  {name}  (already H265)')
        # Copy it to output unchanged so it's available there too
        import shutil
        shutil.copy2(src, dst)
        return True

    print(f'[START] {name}  ({codec} → H265)')

    cmd = [
        'ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error',
        '-stats',
        '-i', src,
        '-c:v', VIDEO_CODEC, *QUALITY_FLAG,
        '-c:a', AUDIO_CODEC,
        '-movflags', '+faststart',
        dst
    ]

    result = subprocess.run(cmd)

    if result.returncode == 0 and os.path.exists(dst) and os.path.getsize(dst) > 0:
        src_mb  = os.path.getsize(src) / 1_048_576
        dst_mb  = os.path.getsize(dst) / 1_048_576
        savings = (1 - dst_mb / src_mb) * 100
        print(f'[DONE]  {name}  {src_mb:.1f} MB → {dst_mb:.1f} MB  ({savings:+.1f}%)')
        return True
    else:
        os.remove(dst) if os.path.exists(dst) else None
        print(f'[FAIL]  {name}  — output missing or ffmpeg error; original kept')
        return False

In [ ]:
# ── Cell 5: Run transcoding ────────────────────────────────────────────────────
import glob, os

files = sorted(
    glob.glob(os.path.join(INPUT_DIR, '**', '*.mkv'), recursive=True) +
    glob.glob(os.path.join(INPUT_DIR, '**', '*.mp4'), recursive=True)
)

print(f'Found {len(files)} file(s) to process\n')

ok, fail = 0, 0
for src in files:
    # Preserve sub-folder structure inside OUTPUT_DIR
    rel  = os.path.relpath(src, INPUT_DIR)
    # Always output as .mp4 (H265 in MP4 container plays everywhere)
    rel_mp4 = os.path.splitext(rel)[0] + '.mp4'
    dst  = os.path.join(OUTPUT_DIR, rel_mp4)
    os.makedirs(os.path.dirname(dst), exist_ok=True)

    if transcode(src, dst):
        ok += 1
    else:
        fail += 1

print(f'\n── Done ── {ok} succeeded, {fail} failed')

In [ ]:
# ── Cell 6 (OPTIONAL): Delete originals after successful transcode ─────────────
# Only run this cell AFTER you've verified the output files look correct!
#
# This deletes files from your Google Drive INPUT folder that have a
# corresponding output file. Uncomment to run.

# import glob, os
# deleted = 0
# for src in files:
#     rel     = os.path.relpath(src, INPUT_DIR)
#     rel_mp4 = os.path.splitext(rel)[0] + '.mp4'
#     dst     = os.path.join(OUTPUT_DIR, rel_mp4)
#     if os.path.exists(dst) and os.path.getsize(dst) > 0:
#         os.remove(src)
#         print(f'[DEL] {os.path.basename(src)}')
#         deleted += 1
# print(f'Deleted {deleted} original(s)')